# Import thư viện cần thiết

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR
import os  # For Kaggle paths

print("✅ All imports done!")

# Load Data

In [ ]:
data_path = "/kaggle/input/digit-recognizer/"

train_df = pd.read_csv(os.path.join(data_path, "train.csv"))
test_df = pd.read_csv(os.path.join(data_path, "test.csv"))  # for Kaggle submission

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Separate X / y

In [ ]:
y = train_df["label"].values
X = train_df.drop(columns=["label"]).values

# Reshape into 28x28 images for DL
X_img = X.reshape(-1, 28, 28)

print("✅ X shape:", X.shape)
print("✅ X_img shape:", X_img.shape)

# Visualization

In [ ]:
plt.figure(figsize=(8, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(X_img[i], cmap="gray")
    plt.title(f"Label: {y[i]}")
    plt.axis("off")
plt.suptitle("Examples from the dataset")
plt.show()

# Distribution Plot

In [ ]:
sns.countplot(x=y)
plt.title("Distribution of digits")
plt.show()

# Normalization

Scale pixel từ 0-255 về 0-1 để mô hình học tốt hơn (CNN yêu cầu input chuẩn hóa).

In [ ]:
X_norm = (X / 255.0).astype("float32")
X_img_norm = (X_img / 255.0).astype("float32")

print("✅ Normalization done! Min/Max:", X_norm.min(), X_norm.max())

# Split train / val

Chia 90% train / 10% val, giữ tỷ lệ nhãn cân bằng (stratify=y).

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_norm, y, test_size=0.1, stratify=y, random_state=42
)

X_train_img, X_val_img, _, _ = train_test_split(
    X_img_norm, y, test_size=0.1, stratify=y, random_state=42
)

print("Train ML:", X_train.shape)
print("Train DL:", X_train_img.shape)
print("Val ML:", X_val.shape)
print("Val DL:", X_val_img.shape)

# Convert to PyTorch Tensors and Create DataLoaders

unsqueeze(1) thêm channel (1 cho grayscale)

TensorDataset + DataLoader để batch (train: 128, val: 256; shuffle train để random)

In [ ]:
# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train_img).unsqueeze(1).float()  # (N,1,28,28)
X_val_t   = torch.tensor(X_val_img).unsqueeze(1).float()
y_train_t = torch.tensor(y_train).long()
y_val_t   = torch.tensor(y_val).long()

# Create datasets and dataloaders
train_ds = TensorDataset(X_train_t, y_train_t)
val_ds   = TensorDataset(X_val_t, y_val_t)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=256)

print("✅ PyTorch DataLoaders created!")
print("Train batches:", len(train_dl))
print("Val batches:", len(val_dl))

# Define CNN Model

- Xây dựng kiến trúc CNN đơn giản cho phân loại ảnh 28x28 (10 lớp).
- Features: 2 block Conv (32->64 filters, kernel 3x3, padding giữ kích thước), ReLU activation, BatchNorm ổn định, MaxPool giảm kích thước (28->14->7), Dropout chống overfit.
- Classifier: Flatten -> Linear (3136->256) + ReLU + BatchNorm + Dropout -> Linear (256->10).
- forward: Chạy qua features rồi classifier.


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.MaxPool2d(2), nn.Dropout(0.5),

            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
            nn.MaxPool2d(2), nn.Dropout(0.5)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.6),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CNN().to(device)

print(f"✅ Model created on {device}!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Training Loop

- Huấn luyện model qua 50 epochs, theo dõi loss và accuracy.
- Loss: CrossEntropy (phù hợp classification).
- Optimizer: Adam (lr=0.001).
- Scheduler: Giảm lr x0.7 mỗi 5 epochs để hội tụ tốt.
- Train: Forward, backward, update weights; tính avg loss.
- Val: No_grad, tính loss và acc (argmax so sánh nhãn).

In [ ]:
loss_fn = nn.CrossEntropyLoss()
opt = optim.Adam(model.parameters(), lr=1e-3)

# Scheduler: divide LR by 0.7 every 5 epochs
scheduler = StepLR(opt, step_size=5, gamma=0.7)

train_losses, val_losses, val_accs = [], [], []

for epoch in range(50):  # Sufficient to exceed 99% accuracy
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    train_losses.append(total_loss / len(train_dl))

    # Validation
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            val_loss += loss_fn(pred, yb).item()
            correct += (pred.argmax(1) == yb).sum().item()
            total += yb.size(0)
    val_losses.append(val_loss / len(val_dl))
    val_accs.append(correct / total)

    # Scheduler step at end of epoch
    scheduler.step()

    # Display
    print(f"Epoch {epoch+1}: "
          f"TrainLoss={train_losses[-1]:.4f}, "
          f"ValLoss={val_losses[-1]:.4f}, "
          f"ValAcc={val_accs[-1]*100:.2f}%, "
          f"LR={scheduler.get_last_lr()[0]:.6f}")

# Plot Training Curves

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.title("Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(val_accs)
plt.title("Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.ylim(0.95, 1.0)

plt.tight_layout()
plt.show()

print("✅ Training complete! Final Val Acc:", f"{val_accs[-1]*100:.2f}%")

- Loss Curves: Train loss giảm mạnh từ 0.23 xuống ~0.007 (hội tụ nhanh sau 5 epochs); val loss ổn định ~0.02 sau epoch 10, sát train loss – không overfitting, nhờ scheduler LR (từ 0.001 xuống 0.000028).
- Validation Accuracy: Tăng từ 98.45% lên đỉnh 99.62% (epoch 14), sau đó plateau ~99.5% – hiệu suất tốt, ổn định cao cho CNN trên Digit Recognizer.
- Tổng Kết: Mô hình học hiệu quả (min val loss 0.0167 tại epoch 13), final acc 99.55%.

# Inference for Kaggle Submission

- Dự đoán trên test set và tạo file CSV cho Kaggle submission.

In [ ]:
# Prepare test data (normalize and reshape in memory)
X_test_norm = (test_df.values / 255.0).astype("float32")
X_test_img = X_test_norm.reshape(-1, 28, 28)
X_test_t = torch.tensor(X_test_img).unsqueeze(1).float()  # (N,1,28,28)
test_dl = DataLoader(X_test_t, batch_size=256)

# Predictions
model.eval()
preds = []
with torch.no_grad():
    for xb in test_dl:
        xb = xb.to(device)
        out = model(xb)
        preds.extend(out.argmax(1).cpu().numpy())

# Create submission
submission = pd.DataFrame({
    "ImageId": np.arange(1, len(preds) + 1),
    "Label": preds
})

output_path = "/kaggle/working/"
submission.to_csv(os.path.join(output_path, "submission_cnn.csv"), index=False)
print("✅ Submission file generated: /kaggle/working/submission_cnn.csv")